# Benchmark HTR/OCR - Manuscrits de Sieyès

Ce notebook évalue la performance de différents modèles de vision-langage sur des images de manuscrits historiques français.

## Structure du notebook

### 📋 **PARTIE 1 : CONFIGURATION**
- **1.1** Imports et dépendances
- **1.2** Configuration des chemins et répertoires
- **1.3** Paramètres du benchmark
- **1.4** Chargement des données (modèles et images)
- **1.5** Fonctions utilitaires

### ⚙️ **PARTIE 2 : EXÉCUTION**
- **2.1** Traitement des images avec les modèles
- **2.2** Sauvegarde des résultats intermédiaires

### 📊 **PARTIE 3 : ANALYSE**
- **3.1** Génération de la table de résultats
- **3.2** Génération des rapports et du viewer interactif
- **3.3** Métriques de performance additionnelles
- **3.4** Export et synthèse

## 📋 PARTIE 1 : CONFIGURATION

> ⚠️ **Note** : Cette section configure l'environnement et charge les données nécessaires.
> Exécutez toutes ces cellules avant de passer à l'exécution du benchmark.

### 1.1 Imports et dépendances

In [ ]:
import os
import json
import random
import concurrent.futures
from functools import partial
from datetime import datetime
from pathlib import Path
import pandas as pd
from tqdm.notebook import tqdm
from api_clients import query_model
from config import system_prompt
from reporting import generate_results_md_table

### 1.2 Configuration des chemins et répertoires

In [ ]:
# Create necessary directories if they don't exist
results_dir = Path("résultats")
reference_dir = Path("transcriptions_de_référence")
images_dir = Path("images")

results_dir.mkdir(exist_ok=True)
reference_dir.mkdir(exist_ok=True)

### 1.3 Paramètres du benchmark

In [ ]:
# Paramètres d'exécution parallèle
MAX_WORKERS = 1  # Nombre de workers pour le traitement parallèle
RANDOM_SEED = 42  # Pour la reproductibilité
SHUFFLE_MODELS = True  # Mélange aléatoire des modèles pour éviter les biais

# Configuration de la seed pour la reproductibilité
random.seed(RANDOM_SEED)

### 1.4 Chargement des données (modèles et images)

In [ ]:
# Load models configuration
with open("models_to_test.json", "r") as f:
    models = json.load(f)

# Get list of images
image_files = list(images_dir.glob("*.jpg")) + list(images_dir.glob("*.png"))

# Apply shuffling if configured
if SHUFFLE_MODELS:
    random.shuffle(models)

print(f"✅ Found {len(image_files)} images to process")
print(f"✅ Will test {len(models)} models")
print(f"✅ Random seed: {RANDOM_SEED}")
print(f"✅ Workers: {MAX_WORKERS}")

### 1.5 Fonctions utilitaires

In [ ]:
# Function to determine model metadata
def get_model_metadata(model_id, model_type):
    """Extract editor and model type from model ID and type"""
    parts = model_id.split("/")
    editor = parts[0]
    
    # Map model type to French
    model_type_fr = "libre" if model_type == "open" else "propriétaire"
    
    return {
        "editeur": editor,
        "modele_type": model_type_fr
    }

## 2. EXÉCUTION DU BENCHMARK

## ⚙️ PARTIE 2 : EXÉCUTION DU BENCHMARK

> ⚠️ **Note** : Cette section lance le traitement des images avec les modèles.
> L'exécution peut prendre plusieurs heures selon le nombre de modèles et d'images.

### 2.1 Traitement des images avec les modèles

#### Fonction de traitement unitaire

In [ ]:
def process_image(img_path, model_info):
    """Traite une image avec un modèle donné"""
    model_meta = get_model_metadata(model_info[0], model_info[1])
    model, _ = model_info
    
    # Create valid filename by replacing invalid characters
    safe_model_name = model.replace('/', '_').replace('\\', '_').replace(':', '_')
    result_file = results_dir / f"{img_path.stem}_{safe_model_name}.json"
    
    # Check if result file already exists
    if result_file.exists():
        return None
        
    try:
        # Query the model
        response_data, cost = query_model(str(img_path), model)

        # Extract the transcription from the response based on format
        if 'choices' in response_data and response_data['choices']:
            transcription = response_data['choices'][0]['message']['content']
        elif 'result' in response_data:
            # This is for Transkribus API format
            transcription = response_data['result']
        else:
            raise Exception(f"Unexpected response format: {response_data}")
        
        # Get usage data with defaults if missing
        usage_data = response_data.get('usage', {})
        if not usage_data or not isinstance(usage_data, dict):
            usage_data = {
                'prompt_tokens': 0,
                'completion_tokens': 0,
                'total_tokens': 0
            }

        # Prepare result data
        result_data = {
            "model": model,
            "editeur": model_meta["editeur"],
            "modele_type": model_meta["modele_type"],
            "image": str(img_path),
            "result": transcription,
            "timestamp": datetime.now().isoformat(),
            "model_info": response_data.get('model_info', {}),
            "usage": usage_data,
            "latency": response_data.get('created') - response_data.get('started', 0) if 'created' in response_data else None
        }
        
        # Save result to file
        with open(result_file, 'w', encoding='utf-8') as f:
            json.dump(result_data, f, ensure_ascii=False, indent=2)
            
        return result_data
        
    except Exception as e:
        print(f"Error processing {img_path} with {model}: {str(e)}")
        return None

### 2.2 Exécution du traitement parallèle

In [ ]:
# Initialize results list
results = []

print(f"🚀 Démarrage du traitement de {len(image_files)} images avec {len(models)} modèles")
print(f"📁 Les résultats seront sauvegardés dans : {results_dir}")

# Process models in parallel
with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    for model_info in tqdm(models, desc="Processing models"):
        # Create partial function with fixed model_info
        process_fn = partial(process_image, model_info=model_info)
        
        # Submit all images for this model to process in parallel
        future_to_img = {executor.submit(process_fn, img_path): img_path 
                        for img_path in image_files}
        
        # Process results as they complete
        for future in tqdm(concurrent.futures.as_completed(future_to_img), 
                         total=len(future_to_img),
                         desc=f"Processing images with {model_info[0]}", 
                         leave=False):
            result = future.result()
            if result is not None:
                results.append(result)

print(f"✅ Traitement terminé : {len(results)} résultats obtenus")

### 3.1 Génération de la table de résultats

In [ ]:
# Generate the results table
generate_results_md_table()
print("✅ Table de résultats générée avec succès")

In [ ]:
# Generate the results table
generate_results_md_table()
print("✅ Table de résultats générée avec succès")

### 3.2 Génération des rapports et du viewer interactif

In [ ]:
# Génération de la table de performance
!python scripts/generate_performance_table.py

In [ ]:
# Génération des données pour le viewer interactif
!python scripts/generate_viewer_data.py

### 3.4 Export et synthèse finale

In [ ]:
# Create a DataFrame for additional analysis
df_results = pd.DataFrame(results)

# Add columns for latency and token usage analysis
if not df_results.empty:
    df_summary = df_results.groupby('model').agg({
        'latency': ['mean', 'min', 'max'],
        'usage': lambda x: pd.Series([d.get('total_tokens', 0) for d in x]).mean()
    }).round(2)
    
    df_summary.columns = ['Avg Latency (s)', 'Min Latency (s)', 'Max Latency (s)', 'Avg Tokens']
    display(df_summary)
else:
    print("Aucun résultat disponible pour l'analyse")

### 3.4 Export et synthèse finale

⚠️ **Actions manuelles après le benchmark** :
1. Régénérer `analyse_modeles.md` pour la synthèse des résultats
2. Mettre à jour le README avec les dernières statistiques  
3. Vérifier le viewer interactif via `python server.py`
4. Committer les résultats si nécessaire

✅ **Le benchmark est maintenant terminé !**